In [1]:
# make and activate venv environment, manually ...

# !pip install torch==2.9.1+cu128 --index-url https://download.pytorch.org/whl/cu128
# !pip install pytorch-lightning~=2.0

# !pip install torch_geometric
# !pip install torch_cluster -f https://data.pyg.org/whl/torch-2.9.1+cu128.html  # no windows version for torch-2.9.1+cu130

# # loading pre-trained model requires numpy < 2
# !pip install -r requirements.txt

# !pip install pysdf>=0.1
# # getting an error like error C2039: 'high_resolution_clock': is not a member of 'std::chrono'?
# # fix these errors by adding "#include <chrono>" to the relevant files or installing this in a VS 2019 tools prompt or older

# # install executorch
# !pip install executorch


# CPU-Only
# !pip install torch 
# !pip install pytorch-lightning 
# !pip install torch_geometric 
# !pip install torch_cluster -f https://data.pyg.org/whl/torch-2.9.1+cpu.html 
# !pip install -r requirements.txt 
# !pip install pysdf@https://vapor.cg.tuwien.ac.at/index.php/s/cPwtcBB95pmLDYw/download/pysdf-0.1.9.tar.gz 
# !pip install executorch


In [2]:
# capture test data for tracing 
# capture manually once while stepping through the code
# add breakpoints and run the code in debug mode, then execute the following lines in the watch window to save the test data for tracing
# relevant code lines are in poco_utils.py, functions generate_latent_representation and predict_from_latent

# encoding:
# torch.save((batch, network, network_latent_size, gen_subsample_manifold_iter, gen_subsample_manifold, None), 'tracing_test_data/generate_latent_representation_in.pt')
# torch.save(batch, 'tracing_test_data/get_data_poco_in.pt')
# torch.save(shape_data_poco, 'tracing_test_data/get_data_poco_out.pt')
# torch.save(data_partial, 'tracing_test_data/network_get_latent_in.pt')
# torch.save(partial_latent, 'tracing_test_data/network_get_latent_out.pt')
# torch.save((shape_data_poco, latent), 'tracing_test_data/generate_latent_representation_out.pt')

# decoding:
# torch.save((latent, network, pts_query, pts_raw_ms, num_pts_local, None), 'tracing_test_data/predict_from_latent_in.pt')
# torch.save(latent, 'tracing_test_data/network_from_latent_in.pt')
# torch.save(occ_hat, 'tracing_test_data/network_from_latent_out.pt')
# torch.save(occ_hat, 'tracing_test_data/predict_from_latent_out.pt')

In [3]:
import torch
import contextlib

# 1. Create a dummy context manager that does literally nothing
@contextlib.contextmanager
def dummy_record_function(*args, **kwargs):
    yield

# 2. Monkey-patch PyTorch's profilers so PyG cannot use them during export
torch.autograd.profiler.record_function = dummy_record_function
torch.profiler.record_function = dummy_record_function
torch.autograd.record_function = dummy_record_function

print("PyTorch profilers disabled to protect ExecuTorch tracing.")

PyTorch profilers disabled to protect ExecuTorch tracing.


In [4]:
# check test data
data_partial = torch.load('tracing_test_data/network_get_latent_in.pt', map_location=torch.device('cpu'))
# -> dict:
# {
#     'pts':        Tensor[1, 3, 10000]     (batch_size, 3d, gen_subsample_manifold),
#     'ids43':      Tensor[1, 156, 1]       (batch_size, gen_subsample_manifold // 64, 1), int64
#     'ids32':      Tensor[1, 625, 1]       (batch_size, gen_subsample_manifold // 16, 1), int64
#     'ids21':      Tensor[1, 2500, 1]      (batch_size, gen_subsample_manifold // 4, 1), int64
#     'ids10':      Tensor[1, 10000, 1]     (batch_size, gen_subsample_manifold, 1), int64
#     'support1':   Tensor[1, 3, 2500]      (batch_size, 3d, gen_subsample_manifold // 4)
#     'support2':   Tensor[1, 3, 625]       (batch_size, 3d, gen_subsample_manifold // 16)
#     'support3':   Tensor[1, 3, 156]       (batch_size, 3d, gen_subsample_manifold // 64)
#     'support4':   Tensor[1, 3, 39]        (batch_size, 3d, gen_subsample_manifold // 256)
#     'ids00':      Tensor[1, 10000, 16]    (batch_size, gen_subsample_manifold, InterpAttentionKHeadsNet.k), int64
#     'ids01':      Tensor[1, 2500, 16]     (batch_size, gen_subsample_manifold // 4, InterpAttentionKHeadsNet.k), int64
#     'ids11':      Tensor[1, 2500, 16]     (batch_size, gen_subsample_manifold // 4, InterpAttentionKHeadsNet.k), int64
#     'ids12':      Tensor[1, 625, 16]      (batch_size, gen_subsample_manifold // 16, InterpAttentionKHeadsNet.k), int64
#     'ids22':      Tensor[1, 625, 16]      (batch_size, gen_subsample_manifold // 16, InterpAttentionKHeadsNet.k), int64
#     'ids23':      Tensor[1, 156, 16]      (batch_size, gen_subsample_manifold // 64, InterpAttentionKHeadsNet.k), int64
#     'ids33':      Tensor[1, 156, 16]      (batch_size, gen_subsample_manifold // 64, InterpAttentionKHeadsNet.k), int64
#     'ids34':      Tensor[1, 39, 16]       (batch_size, gen_subsample_manifold // 256, InterpAttentionKHeadsNet.k), int64
#     'ids44':      Tensor[1, 39, 16]       (batch_size, gen_subsample_manifold // 256, InterpAttentionKHeadsNet.k), int64
#     'latents':    Tensor[1, 256, 10000]   (batch_size, network_latent_size, gen_subsample_manifold), float16
# }
partial_latent = torch.load('tracing_test_data/network_get_latent_out.pt', map_location=torch.device('cpu'))
# -> Tensor[1, 256, 10000] (batch_size, network_latent_size, gen_subsample_manifold)

latents = torch.load('tracing_test_data/network_from_latent_in.pt', map_location=torch.device('cpu'))
# -> dict:
# {
#     'pts_ms':             Tensor[1, 59979, 3]     (batch_size, num_points, 3d)
#     'normals_ms':         Tensor[1, 59979, 3]     (batch_size, num_points, 3d), float64
#     'pc_file_in':         str
#     'pts_query_ms':       Tensor[1, 0, 3]         (batch_size, num_query_points, 3d)
#     'imp_surf_dist_ms':   Tensor[1, 0, 3]         (batch_size, num_query_points, 3d)
#     'shape_id':           Tensor[1]               (batch_size, num_query_points, 3d), int64
#     'pts_raw_ms':         Tensor[1, 59979, 3]     (batch_size, num_points, 3d)
#     'pts':                Tensor[1, 3, 59979]     (batch_size, 3d, num_points)
#     'pts_query':          Tensor[1, 50000, 3]     (batch_size, num_query_points, 3d)
#     'occ':                Tensor[1, 0, 3]         (batch_size, num_query_points, 3d), int64
#     'ids43':              Tensor[1, 937, 1]       (batch_size, num_points // 64, 3d), int64
#     'ids32':              Tensor[1, 3748, 1]      (batch_size, num_points // 16, 3d), int64
#     'ids21':              Tensor[1, 14994, 1]     (batch_size, num_points // 4, 3d), int64
#     'ids10':              Tensor[1, 59979, 1]     (batch_size, num_points, 3d), int64
#     'support1':           Tensor[1, 3, 14994]     (batch_size, 3d, num_points // 4)
#     'support2':           Tensor[1, 3, 3748]      (batch_size, 3d, num_points // 16)
#     'support3':           Tensor[1, 3, 937]       (batch_size, 3d, num_points // 64)
#     'support4':           Tensor[1, 3, 234]       (batch_size, 3d, num_points // 256)
#     'ids00':              Tensor[1, 59979, 16]    (batch_size, num_query_points, InterpAttentionKHeadsNet.k), int64
#     'ids01':              Tensor[1, 14994, 16]    (batch_size, num_query_points // 4, InterpAttentionKHeadsNet.k), int64
#     'ids11':              Tensor[1, 14994, 16]    (batch_size, num_query_points // 4, InterpAttentionKHeadsNet.k), int64
#     'ids12':              Tensor[1, 3748, 16]     (batch_size, num_query_points // 16, InterpAttentionKHeadsNet.k), int64
#     'ids22':              Tensor[1, 3748, 16]     (batch_size, num_query_points // 16, InterpAttentionKHeadsNet.k), int64
#     'ids23':              Tensor[1, 937, 16]      (batch_size, num_query_points // 64, InterpAttentionKHeadsNet.k), int64
#     'ids33':              Tensor[1, 937, 16]      (batch_size, num_query_points // 64, InterpAttentionKHeadsNet.k), int64
#     'ids34':              Tensor[1, 234, 16]      (batch_size, num_query_points // 256, InterpAttentionKHeadsNet.k), int64
#     'ids44':              Tensor[1, 234, 16]      (batch_size, num_query_points // 256, InterpAttentionKHeadsNet.k), int64
#     'proj_ids':           Tensor[1, 50000, 64]    (batch_size, rec_batch_size, PocoModel.k), int64
#     'latents':            Tensor[1, 256, 59979]   (batch_size, network_latent_size, num_points)
#     'pts_local_ps':       Tensor[1, 50000, 50, 3] (batch_size, rec_batch_size, num_pts_local, 3d)
# }
# required for POCO inference: latents, proj_ids, pts, pts_query
# required for PPS inference:  pts_local_ps
# only train/val: pts_query_ms, imp_surf_dist_ms, occ
# POCO/PPS reconstruction and bookkeeping: 
# - pc_file_in: str, single point cloud file or dataset dir
# - shape_id: always 0 with batch_size 1
# - pts_raw_ms: point cloud without normalization
# Not used here: normals_ms (maybe in the future), ids and support (just there from the latent generation)
occ_hat = torch.load('tracing_test_data/network_from_latent_out.pt', map_location=torch.device('cpu'))
# -> Tensor[1, 2, 50000] (batch_size, in_out_probabilities, rec_batch_size)

# print key, type and shape if applicable for all items
print('data_partial:')
for key, value in data_partial.items():
    print_str = f'  {key}: {type(value)}'
    if isinstance(value, torch.Tensor):
        print_str += f', shape: {value.shape}, dtype: {value.dtype}'
        if value.numel() > 0:
            print_str += f', min: {value.min()}, max: {value.max()}'
    print(print_str)

print('latent:')
for key, value in latents.items():
    print_str = f'  {key}: {type(value)}'
    if isinstance(value, torch.Tensor):
        print_str += f', shape: {value.shape}, dtype: {value.dtype}'
        if value.numel() > 0:
            print_str += f', min: {value.min()}, max: {value.max()}'
    print(print_str)

data_partial:
  pts: <class 'torch.Tensor'>, shape: torch.Size([1, 3, 10000]), dtype: torch.float32, min: -0.4674682021141052, max: 0.4736515283584595
  ids43: <class 'torch.Tensor'>, shape: torch.Size([1, 156, 1]), dtype: torch.int64, min: 0, max: 38
  ids32: <class 'torch.Tensor'>, shape: torch.Size([1, 625, 1]), dtype: torch.int64, min: 0, max: 155
  ids21: <class 'torch.Tensor'>, shape: torch.Size([1, 2500, 1]), dtype: torch.int64, min: 0, max: 624
  ids10: <class 'torch.Tensor'>, shape: torch.Size([1, 10000, 1]), dtype: torch.int64, min: 0, max: 2499
  support1: <class 'torch.Tensor'>, shape: torch.Size([1, 3, 2500]), dtype: torch.float32, min: -0.4674682021141052, max: 0.4734063148498535
  support2: <class 'torch.Tensor'>, shape: torch.Size([1, 3, 625]), dtype: torch.float32, min: -0.45039722323417664, max: 0.4532862901687622
  support3: <class 'torch.Tensor'>, shape: torch.Size([1, 3, 156]), dtype: torch.float32, min: -0.4201790988445282, max: 0.43585532903671265
  support4: <cl

In [5]:
# # get hparams
# from source.ppsurf_model import PPSurfModel

# import yaml
# import inspect

# with open('models/ppsurf_50nn/version_0/config.yaml', 'r') as f:
#     config = yaml.safe_load(f)

# model_params = config.get('model', config)
# print(model_params)

# # get only the necessary ones
# sig = inspect.signature(PPSurfModel.__init__)
# valid_args = sig.parameters.keys()
# print(valid_args)
# filtered_config = {k: v for k, v in model_params.items() if k in valid_args}
# print(filtered_config)


In [6]:
# load model
from source.ppsurf_model import PPSurfModel

model_kwargs = {
    'pointnet_latent_size': 256,  # Replace with your actual values
    'output_names': ['imp_surf_sign',],
    'in_channels': 3,
    'out_channels': 2,
    'k': 64,
    'lambda_l1': 0.0,
    'debug': False,
    'in_file': 'datasets/abc_minimal/04_pts_vis/00010009_d97409455fa543b3a224250f_trimesh_000.xyz.ply',
    'results_dir': './results_exporter',
    'padding_factor': 0.05,
    'name': 'ppsurf_50nn',
    'network_latent_size': 256,
    'gen_subsample_manifold_iter': 10,
    'gen_subsample_manifold': 10000,
    'gen_resolution_global': 257,
    'num_pts_local': 50,
    'rec_batch_size': 50000,
    'gen_refine_iter': 10,
    'workers': 8
}

model = PPSurfModel.load_from_checkpoint(
    'models/ppsurf_50nn/version_0/checkpoints/last.ckpt',
    **model_kwargs)
model.eval()

print(model)

InterpNet - Simple - K=64
Network -- backbone -- 12798516 parameters
Network -- projection -- 280898 parameters
InterpNet - Simple - K=64
Network -- backbone -- 12798516 parameters
Network -- projection -- 346176 parameters
Network -- point_net -- 471297 parameters
Network -- mlp -- 133122 parameters
PPSurfModel(
  (network): PPSurfNetwork(
    (encoder): FKAConvNetwork(
      (cv0): FKAConvLayer(
        (cv): Conv2d(3, 64, kernel_size=(1, 16), stride=(1, 1), bias=False)
        (fc1): Conv2d(3, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (fc2): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (fc3): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): InstanceNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
        (bn2): InstanceNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
        (activation): SiLU()
      )
      (bn0): BatchNorm1d(64, eps=1e-05, momentum=0.1, affi

In [7]:
import gc
import torch
from torch.export import export, draft_export, Dim
from executorch.exir import to_edge_transform_and_lower

# 1. Grab the internal network from your loaded Lightning model
network = model.network.cpu().eval()
device = network.device
print(f"Network device: {device}")

# 2. Define Wrappers for the two specific methods
class GetLatentWrapper(torch.nn.Module):
    def __init__(self, net):
        super().__init__()
        self.net = net
    def forward(self, pts, ids43, ids32, ids21, ids10, support1, support2, support3, support4, 
                ids00, ids01, ids11, ids12, ids22, ids23, ids33, ids34, ids44):
        
        # Reconstruct the dict internally because of tracing requirements
        data_dict = {
            'pts': pts.clone(), 'ids43': ids43.clone(), 'ids32': ids32.clone(), 'ids21': ids21.clone(), 'ids10': ids10.clone(),
            'support1': support1.clone(), 'support2': support2.clone(), 'support3': support3.clone(), 'support4': support4.clone(),
            'ids00': ids00.clone(), 'ids01': ids01.clone(), 'ids11': ids11.clone(), 'ids12': ids12.clone(), 'ids22': ids22.clone(),
            'ids23': ids23.clone(), 'ids33': ids33.clone(), 'ids34': ids34.clone(), 'ids44': ids44.clone(),
        }
        
        return self.net.encoder.forward(data_dict, spectral_only=True)

class FromLatentWrapper(torch.nn.Module):
    def __init__(self, net):
        super().__init__()
        self.net = net
    def forward(self, latents, proj_ids, pts, pts_query, pts_local_ps):
        # Reconstruct the dict for from_latent
        data_dict = {
            'latents': latents.clone(),
            'proj_ids': proj_ids.clone(),
            'pts': pts.clone(),
            'pts_query': pts_query.clone(),
            'pts_local_ps': pts_local_ps.clone(),
                    }
        occ_probs = self.net.from_latent_proj_ids(data_dict, has_proj_ids=True)
        occ_probs = occ_probs.contiguous().clone()
        return occ_probs

# 3. Define Example Inputs (Match your model's expected shapes)
num_pts_subsample = 10000  # gen_subsample_manifold
num_query_points = 1  # rec_batch_size

# Define the absolute maximums for memory allocation
N_raw = Dim("num_pts_raw", min=1, max=100000)
N_raw_4 = Dim("num_pts_raw_4", min=1, max=100000 // 4)
N_raw_16 = Dim("num_pts_raw_16", min=1, max=100000 // 16)
N_raw_64 = Dim("num_pts_raw_64", min=1, max=100000 // 64)
N_raw_256 = Dim("num_pts_raw_256", min=1, max=100000 // 256)

example_input_get_latent = {
    'pts': torch.randn(1, 3, num_pts_subsample, device=device),
    'ids43': torch.randint(0, num_pts_subsample // 256, (1, num_pts_subsample // 64, 1), device=device),
    'ids32': torch.randint(0, num_pts_subsample // 64, (1, num_pts_subsample // 16, 1), device=device),
    'ids21': torch.randint(0, num_pts_subsample // 16, (1, num_pts_subsample // 4, 1), device=device),
    'ids10': torch.randint(0, num_pts_subsample // 4, (1, num_pts_subsample, 1), device=device),
    'support1': torch.randn(1, 3, num_pts_subsample // 4, device=device),
    'support2': torch.randn(1, 3, num_pts_subsample // 16, device=device),
    'support3': torch.randn(1, 3, num_pts_subsample // 64, device=device),
    'support4': torch.randn(1, 3, num_pts_subsample // 256, device=device),
    'ids00': torch.randint(0, num_pts_subsample, (1, num_pts_subsample, 16), device=device),
    'ids01': torch.randint(0, num_pts_subsample, (1, num_pts_subsample // 4, 16), device=device),
    'ids11': torch.randint(0, num_pts_subsample // 4, (1, num_pts_subsample // 4, 16), device=device),
    'ids12': torch.randint(0, num_pts_subsample // 4, (1, num_pts_subsample // 16, 16), device=device),
    'ids22': torch.randint(0, num_pts_subsample // 16, (1, num_pts_subsample // 16, 16), device=device),
    'ids23': torch.randint(0, num_pts_subsample // 16, (1, num_pts_subsample // 64, 16), device=device),
    'ids33': torch.randint(0, num_pts_subsample // 64, (1, num_pts_subsample // 64, 16), device=device),
    'ids34': torch.randint(0, num_pts_subsample // 64, (1, num_pts_subsample // 256, 16), device=device),
    'ids44': torch.randint(0, num_pts_subsample // 256, (1, num_pts_subsample // 256, 16), device=device),
}

get_latent_keys_order = [
    'pts', 'ids43', 'ids32', 'ids21', 'ids10', 'support1', 'support2', 'support3', 'support4',
    'ids00', 'ids01', 'ids11', 'ids12', 'ids22', 'ids23', 'ids33', 'ids34', 'ids44',
]
args_get_latent = tuple(example_input_get_latent[key] for key in get_latent_keys_order)


num_pts_raw = 59979  # from the test data
# num_query_points = 50000  # rec_batch_size

example_input_from_latent = {
    'latents': torch.randn(1, 256, num_pts_raw, device=device), # Latents output from encoder is always 10000!
    'proj_ids': torch.randint(0, num_pts_subsample, (1, num_query_points, 64), device=device),
    'pts': torch.randn(1, 3, num_pts_raw, device=device),
    'pts_query': torch.randn(1, num_query_points, 3, device=device),
    'pts_local_ps': torch.randn(1, num_query_points, 50, 3, device=device), 
}
from_latent_keys_order = ['latents', 'proj_ids', 'pts', 'pts_query', 'pts_local_ps']
args_from_latent = tuple(example_input_from_latent[k] for k in from_latent_keys_order)

# Define the dynamic dimensions for the real fluctuating sizes
N_raw = Dim("num_pts_raw", min=1, max=100000)
# N_query = Dim("num_query_points", min=1, max=100000)
# Map them to from_latent inputs
dynamic_shapes_from_latent = (
    {2: N_raw},         # [0] latents: (1, 256, num_pts_raw) -> strictly static 10000
    None, # [1] proj_ids: (1, num_query_points, 64)
    {2: N_raw},   # [2] pts: (1, 3, num_pts_raw)
    None, # [3] pts_query: (1, num_query_points, 3)
    None  # [4] pts_local_ps: (1, num_query_points, 50, 3)
)


# 4. Capture and Lower
# We use the new 2026 standard flow: Export -> To Edge/Lower -> To Executorch
def save_pte(wrapper_instance, example_args, dynamic_shapes_tuple, filename):
    # Step A: Capture into ATen Dialect
    exported_program = export(wrapper_instance, example_args, dynamic_shapes=dynamic_shapes_tuple)
    # exported_program = draft_export(wrapper_instance, example_args, dynamic_shapes=dynamic_shapes_tuple)  # for debug info
    
    # Step B: Lower to Edge Dialect and convert to PTE
    # This is the modern replacement for 'capture_program' or 'to_edge'
    pte = to_edge_transform_and_lower(exported_program).to_executorch()
    
    with open(filename, "wb") as f:
        f.write(pte.buffer)
    print(f"Successfully saved {filename}")
    
    # clean up
    del exported_program
    del pte
    gc.collect()

# Execute the saves
# If it fails with 'invalid argument', the file is blocked by the OS. Restart the kernel.
with torch.no_grad():
    save_pte(GetLatentWrapper(network), args_get_latent, None, "get_latent.pte")
    save_pte(FromLatentWrapper(network), args_from_latent, dynamic_shapes_from_latent, "from_latent.pte")

Skipping import of cpp extensions due to incompatible torch version 2.10.0+cpu for torchao version 0.15.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
W0318 14:52:49.716000 7528 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Network device: cpu


C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.2800.0_x64__qbz5n2kfra8p0\Lib\copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.2800.0_x64__qbz5n2kfra8p0\Lib\copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


Successfully saved get_latent.pte


C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.2800.0_x64__qbz5n2kfra8p0\Lib\copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.2800.0_x64__qbz5n2kfra8p0\Lib\copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


Successfully saved from_latent.pte


In [8]:
# load executorch models and compare with original PyTorch outputs
import torch
from executorch.runtime import Runtime

runtime = Runtime.get()

def print_pte_outputs(pte_filename, original_pytorch_output, flattened_inputs_tuple):
    print(f"\n{'='*40}")
    print(f" Inspecting: {pte_filename}")
    print(f"{'='*40}")
    
    # 1. Load and execute
    program = runtime.load_program(pte_filename)
    method = program.load_method("forward")
    contiguous_inputs = [t.contiguous() for t in flattened_inputs_tuple]
    
    et_outputs = method.execute(contiguous_inputs)
    
    # 2. Print ExecuTorch Outputs
    print(f"\n--- EXECUTORCH RETURNED {len(et_outputs)} ITEMS ---")
    for i, out in enumerate(et_outputs):
        if isinstance(out, torch.Tensor):
            print(f"  [{i}] {out.dtype} | Shape: {list(out.shape)}")
            # Print the first 4 numbers to easily spot-check the math
            print(f"      Snippet: {out.flatten()[:4].tolist()}")
        else:
            print(f"  [{i}] Non-Tensor type: {type(out)}")

    # 3. Print PyTorch Original Outputs
    # Standardize to a list for easy iteration
    if isinstance(original_pytorch_output, torch.Tensor):
        py_outs = [original_pytorch_output]
    else:
        py_outs = original_pytorch_output
        
    print(f"\n--- PYTORCH RETURNED {len(py_outs)} ITEMS ---")
    for i, out in enumerate(py_outs):
        if isinstance(out, torch.Tensor):
            print(f"  [{i}] {out.dtype} | Shape: {list(out.shape)}")
            print(f"      Snippet: {out.flatten()[:4].tolist()}")
        else:
            print(f"  [{i}] Non-Tensor type: {type(out)}")
            
    print("\n")

# --- EXECUTE ---
with torch.no_grad():
    print("Running PyTorch baselines...")
    py_get_latent_out = GetLatentWrapper(network)(*args_get_latent)
    py_from_latent_out = FromLatentWrapper(network)(*args_from_latent)
    
    print_pte_outputs("get_latent.pte", py_get_latent_out, args_get_latent)
    print_pte_outputs("from_latent.pte", py_from_latent_out, args_from_latent)

Running PyTorch baselines...


[C:\actions-runner\_work\executorch\executorch\et\executorch\runtime\executor\program.cpp:154] InternalConsistency verification requested but not available
[C:\actions-runner\_work\executorch\executorch\et\executorch\extension\threadpool\cpuinfo_utils.cpp:71] Reading file /sys/devices/soc0/image_version
[C:\actions-runner\_work\executorch\executorch\et\executorch\extension\threadpool\cpuinfo_utils.cpp:87] Failed to open midr file /sys/devices/soc0/image_version
[C:\actions-runner\_work\executorch\executorch\et\executorch\extension\threadpool\cpuinfo_utils.cpp:100] Reading file /sys/devices/system/cpu/cpu0/regs/identification/midr_el1
[C:\actions-runner\_work\executorch\executorch\et\executorch\extension\threadpool\cpuinfo_utils.cpp:109] Failed to open midr file /sys/devices/system/cpu/cpu0/regs/identification/midr_el1
[C:\actions-runner\_work\executorch\executorch\et\executorch\extension\threadpool\cpuinfo_utils.cpp:125] CPU info and manual query on # of cpus dont match.



 Inspecting: get_latent.pte

--- EXECUTORCH RETURNED 1 ITEMS ---
  [0] torch.float32 | Shape: [1, 256, 10000]
      Snippet: [2.0003345012664795, 1.615902304649353, 2.8511526584625244, -0.6387262940406799]

--- PYTORCH RETURNED 1 ITEMS ---
  [0] torch.float32 | Shape: [1, 256, 10000]
      Snippet: [2.0032310485839844, 1.6185414791107178, 2.8509888648986816, -0.6368224620819092]




[C:\actions-runner\_work\executorch\executorch\et\executorch\runtime\executor\program.cpp:154] InternalConsistency verification requested but not available



 Inspecting: from_latent.pte

--- EXECUTORCH RETURNED 1 ITEMS ---
  [0] torch.float32 | Shape: [1, 2, 1]
      Snippet: [18.749698638916016, -17.74637794494629]

--- PYTORCH RETURNED 1 ITEMS ---
  [0] torch.float32 | Shape: [1, 2, 1]
      Snippet: [18.749692916870117, -17.74637222290039]




In [9]:
import time
import numpy as np
import trimesh
import torch
from executorch.runtime import Runtime
from source.poco_data_loader import get_fkaconv_ids

device_test = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

runtime = Runtime.get()
get_latent_program = runtime.load_program("get_latent.pte")
get_latent_method = get_latent_program.load_method("forward")

# 1. Load raw point cloud
pc = trimesh.load('datasets/abc_minimal/04_pts_vis/00010009_d97409455fa543b3a224250f_trimesh_000.xyz.ply')
pts_raw = np.array(pc.vertices)  # [num_pts_raw, 3] 
num_pts_raw = pts_raw.shape[0]

# 2. Setup TTA Parameters
num_pts_subsample = 10000 
target_coverage = 3

print(f"Starting Guaranteed TTA: Target >= {target_coverage}x coverage for all {num_pts_raw} points.")

# 3. Initialize Global Accumulators
global_latents = torch.zeros((1, 256, num_pts_raw), device=device_test)
# Flatten counts for easier logic/indexing
counts = torch.zeros((num_pts_raw,), dtype=torch.int32, device=device_test) 

start_time = time.time()
iteration = 0

# 4. TTA Loop (Runs until the least-sampled point hits the target)
while counts.min().item() < target_coverage:
    iter_start = time.time()
    
    # Move counts to numpy for fast filtering
    current_counts = counts.cpu().numpy()
    
    # Identify which points still need to be sampled
    deficient_indices = np.where(current_counts < target_coverage)[0]
    
    if len(deficient_indices) >= num_pts_subsample:
        # We have plenty of under-sampled points. Grab 10k of them randomly.
        np.random.shuffle(deficient_indices)
        sub_idx = deficient_indices[:num_pts_subsample]
    else:
        # We are at the very end! Take all remaining deficient points...
        needed = num_pts_subsample - len(deficient_indices)
        
        # ...and pad the rest of the 10k batch with points we've already covered
        sufficient_indices = np.where(current_counts >= target_coverage)[0]
        fill_idx = np.random.choice(sufficient_indices, size=needed, replace=False)
        sub_idx = np.concatenate([deficient_indices, fill_idx])
        
    pts_sub = pts_raw[sub_idx]
    
    # Prepare data dictionary
    pts_tensor = torch.from_numpy(pts_sub).float().unsqueeze(0).transpose(1, 2).to(device_test)
    data_encoding = {'pts': pts_tensor}
    
    fkaconv_ids = get_fkaconv_ids(data_encoding)
    data_encoding.update(fkaconv_ids)
    
    # Ensure contiguous memory for ExecuTorch
    data_encoding_list = [t.contiguous() for t in data_encoding.values()]
    
    # Execute model
    latents_out = get_latent_method.execute(data_encoding_list)[0]
    
    # Ensure output is a tensor on the correct device
    if not isinstance(latents_out, torch.Tensor):
        latents_out = torch.tensor(latents_out)
    latents_out = latents_out.to(device_test)
    
    # Accumulate into global tensors
    global_latents[0, :, sub_idx] += latents_out[0]
    counts[sub_idx] += 1
    
    iter_time = time.time() - iter_start
    iteration += 1
    
    # Print live stats so you can watch the coverage fill up
    min_c = counts.min().item()
    mean_c = counts.float().mean().item()
    max_c = counts.max().item()
    print(f"  Pass {iteration} | {iter_time:.4f}s | Coverage (Min: {min_c}, Mean: {mean_c:.2f}, Max: {max_c})")

# 5. Finalize and Average
# Division is perfectly safe now because we guaranteed min count >= target_coverage
final_latents = global_latents / counts.view(1, 1, num_pts_raw)

end_time = time.time()
total_elapsed = end_time - start_time

print(f'\nEncoding Guaranteed TTA completed in {total_elapsed:.4f} seconds ({iteration} passes).')
print(f'Average time per pass: {total_elapsed / iteration:.4f} seconds.')
print(f'Final Global Latent shape: {final_latents.shape}, dtype: {final_latents.dtype}')

# save final latents for later use
final_latents_file = 'final_latents.pt'
torch.save(final_latents.cpu(), final_latents_file)

[C:\actions-runner\_work\executorch\executorch\et\executorch\runtime\executor\program.cpp:154] InternalConsistency verification requested but not available


Starting Guaranteed TTA: Target >= 3x coverage for all 60031 points.


c:\repos\pps\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  Pass 1 | 8.6298s | Coverage (Min: 0, Mean: 0.17, Max: 1)
  Pass 2 | 8.0311s | Coverage (Min: 0, Mean: 0.33, Max: 2)
  Pass 3 | 8.4475s | Coverage (Min: 0, Mean: 0.50, Max: 3)
  Pass 4 | 8.1874s | Coverage (Min: 0, Mean: 0.67, Max: 3)
  Pass 5 | 7.9989s | Coverage (Min: 0, Mean: 0.83, Max: 3)
  Pass 6 | 8.2017s | Coverage (Min: 0, Mean: 1.00, Max: 3)
  Pass 7 | 8.4026s | Coverage (Min: 0, Mean: 1.17, Max: 3)
  Pass 8 | 8.8257s | Coverage (Min: 0, Mean: 1.33, Max: 3)
  Pass 9 | 8.8753s | Coverage (Min: 0, Mean: 1.50, Max: 3)
  Pass 10 | 8.9848s | Coverage (Min: 0, Mean: 1.67, Max: 3)
  Pass 11 | 9.0569s | Coverage (Min: 0, Mean: 1.83, Max: 3)
  Pass 12 | 9.3047s | Coverage (Min: 0, Mean: 2.00, Max: 3)
  Pass 13 | 9.1924s | Coverage (Min: 0, Mean: 2.17, Max: 3)
  Pass 14 | 8.9208s | Coverage (Min: 0, Mean: 2.33, Max: 3)
  Pass 15 | 8.9177s | Coverage (Min: 0, Mean: 2.50, Max: 3)
  Pass 16 | 8.9658s | Coverage (Min: 0, Mean: 2.67, Max: 3)
  Pass 17 | 8.7633s | Coverage (Min: 0, Mean: 2.8

In [10]:
# test decoding (signed distance prediction) with proper inputs and time results
import time
import tqdm
import numpy as np
import trimesh
import torch
import torch.nn.functional as func
from executorch.runtime import Runtime
from source.poco_utils import get_pts_local_ps
from source.poco_data_loader import get_proj_ids
from source.base.proximity import make_kdtree
from source.base.point_cloud import write_ply

project_k = 64  # from poco config > model > init_args > k
num_pts_local = 50  # from pps config > model > init_args > num_pts_local

device_test = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
runtime = Runtime.get()
from_latent_program = runtime.load_program("from_latent.pte")
from_latent_method = from_latent_program.load_method("forward")

final_latents_file = 'final_latents.pt'
final_latents = torch.load(final_latents_file)

# Load raw point cloud
pc = trimesh.load('datasets/abc_minimal/04_pts_vis/00010009_d97409455fa543b3a224250f_trimesh_000.xyz.ply')
pts_raw = np.array(pc.vertices)  # [num_pts_raw, 3] 
num_pts_raw = pts_raw.shape[0]
kdtree = make_kdtree(pts=pts_raw)

# make query points for a grid
grid_res = 4
pts_query_np = np.stack(np.meshgrid(
    np.linspace(-0.5, 0.5, grid_res),
    np.linspace(-0.5, 0.5, grid_res),
    np.linspace(-0.5, 0.5, grid_res),
 ))
num_query_points = grid_res ** 3
pts_query_np_flat = np.transpose(pts_query_np, (1, 2, 3, 0)).reshape(-1, 3)  # [num_query_points, 3]

# prepare input for from_latent
inputs_from_latent = {
    'latents': final_latents,  # [1, latent_size, num_pts_raw]
    'proj_ids': None,  # [1, num_query_points, project_k], filled per query point
    'pts': torch.from_numpy(pts_raw).float().unsqueeze(0).transpose(1, 2).to(device_test),  # [1, 3, num_pts_raw]
    'pts_ms': torch.from_numpy(pts_raw).float().unsqueeze(0).to(device_test),  # [1, num_pts_raw, 3], only for local subsamples
    'pts_query': None,  # [1, num_query_points, 3], filled per query point
    'pts_local_ps': None,  # [1, num_query_points, num_pts_local, 3], filled per query point
}

# result buffer
occ_hat = torch.zeros((num_query_points, 2), device=device_test)  # [num_query_points, in_out_probabilities]

# estimate the signed distance for every query point
start_time = time.time()
for pti, pt_query in enumerate(tqdm.tqdm(pts_query_np_flat)):
    
    # update input dict for this query point
    inputs_from_latent['pts_query'] = torch.from_numpy(pt_query).float().unsqueeze(0).unsqueeze(0).to(device_test)  # [1, 1, 3]
    pts_local_ps = get_pts_local_ps(inputs_from_latent, pt_query[None, :], kdtree, pts_raw, num_pts_local)
    inputs_from_latent['pts_local_ps'] = pts_local_ps.float().to(device_test) # [1, num_pts_local, 3]
    
    spatial_data = get_proj_ids(inputs_from_latent, project_k)
    inputs_from_latent['proj_ids'] = spatial_data['proj_ids']
    
    inputs_list = [inputs_from_latent[k].contiguous() for k in ['latents', 'proj_ids', 'pts', 'pts_query', 'pts_local_ps']]
    occ_probs = from_latent_method.execute(inputs_list)[0]  # [1, 2, 1]
    occ_hat[pti] = occ_probs.squeeze(0).squeeze(1)  # [2]

end_time = time.time()
total_elapsed = end_time - start_time
print(f'\nDecoding loop completed in {total_elapsed:.4f} seconds.')
print(f'Average time per query point: {total_elapsed / num_query_points:.6f} seconds.')

occ_hat = func.softmax(occ_hat, dim=1)
occ_hat = occ_hat[:, 0] - occ_hat[:, 1]
occ_hat = occ_hat.squeeze(0).detach().cpu().numpy()
print(f"Signed distance [{occ_hat.shape}] snippet: {occ_hat[:10]}")

[C:\actions-runner\_work\executorch\executorch\et\executorch\runtime\executor\program.cpp:154] InternalConsistency verification requested but not available
100%|██████████| 64/64 [00:09<00:00,  6.88it/s]


Decoding loop completed in 9.3016 seconds.
Average time per query point: 0.145337 seconds.
Signed distance [(64,)] snippet: [0.99999946 0.9999993  0.99999833 0.99998975 0.9999997  1.
 1.         0.9999978  0.9999983  1.        ]


In [11]:
# visualize results to verify
# reshape back to grid for visualization, write as point cloud, green for in, red for out
pts_query_vis = pts_query_np.reshape(3, -1).transpose(0, 1)  # [num_query_points, 3]
colors = np.zeros((num_query_points, 3), dtype=np.uint8)
colors[occ_hat > 0] = [0, 255, 0]  # Green for in
colors[occ_hat <= 0] = [255, 0, 0]  # Red for out
write_ply('tracing_test_data/decoded_points.ply', pts_query_vis, colors=colors)

# show center slice
print(occ_hat.reshape(grid_res, grid_res, grid_res)[:, grid_res//2, :])

[[ 0.9999983   1.          1.          0.99999905]
 [ 1.         -0.9072131  -0.89181316  1.        ]
 [ 1.         -0.89701116 -0.9179267   1.        ]
 [ 0.9999912   0.99999994  0.9999993   0.99999857]]
